In [1]:
from abc import ABC, abstractmethod
import pyspark.sql.functions as F

StatementMeta(, 27f4e40f-3b08-4fe3-a1ea-e6b2accfe71d, 3, Finished, Available, Finished, False)

In [1]:
class Table(ABC):

    @property
    @abstractmethod
    def STORAGE_NAME(self) -> str:
        pass
    
    @property
    @abstractmethod
    def TABLE_NAME(self) -> str:
        pass
    
    def __init__(self):
        self.df = self.read()

    def read(self):
        return spark.read.table(f"{self.STORAGE_NAME}.dbo.{self.TABLE_NAME}")
        
    def get_agg_by_column(self, agg_func, column):
        return self.df.select(F.col(column).alias("column"))\
            .agg(agg_func("column").alias("res"))\
            .first()["res"]

StatementMeta(, 6950b3b7-66f8-4e40-be86-bfeeb9d313e8, 3, Finished, Available, Finished, True)

NameError: name 'ABC' is not defined

In [4]:
class TableWithDate(ABC):

    @property
    @abstractmethod
    def DATE_COLUMN(self) -> str:
        pass

StatementMeta(, 27f4e40f-3b08-4fe3-a1ea-e6b2accfe71d, 6, Finished, Available, Finished, False)

In [ ]:
class TableWithPKS(ABC):

    @property
    @abstractmethod
    def PKS(self) -> list[str]:
        pass
    
    @abstractmethod
    def append(self, new_rows):
        pass

    def get_unique(self):
        return self.df.select(*self.PKS).distinct()
        
    def get_new_rows(self, new_df):
        current_rows = self.get_unique()
        return new_df.join(current_rows, on=self.PKS, how="left_anti")

    def insert_new(self, new_df):
        new_rows = self.get_new_rows(new_df)
        self.append(new_rows)


In [3]:
class SilverTable(Table, ABC):

    STORAGE_NAME = "Silver_lakehouse"

StatementMeta(, 27f4e40f-3b08-4fe3-a1ea-e6b2accfe71d, 5, Finished, Available, Finished, False)

In [5]:
class TaxiSilverTable(SilverTable, TableWithDate, ABC):

    DATE_COLUMN = "pickup_datetime"

    @property
    @abstractmethod
    def TAXI_TYPE_ID(self) -> int:
        pass

StatementMeta(, 27f4e40f-3b08-4fe3-a1ea-e6b2accfe71d, 7, Finished, Available, Finished, False)

In [6]:
class YellowTaxiTable(TaxiSilverTable):
    
    TABLE_NAME = "yellow_tripdata"
    TAXI_TYPE_ID = 1

StatementMeta(, 27f4e40f-3b08-4fe3-a1ea-e6b2accfe71d, 8, Finished, Available, Finished, False)

In [ ]:
class GreenTaxiTable(TaxiSilverTable):
    
    TABLE_NAME = "green_tripdata"
    TAXI_TYPE_ID = 2

In [ ]:
class FHVTaxiTable(TaxiSilverTable):
    
    TABLE_NAME = "fhv_tripdata"
    TAXI_TYPE_ID = 3

In [ ]:
class FHVHVTaxiTable(TaxiSilverTable):
    
    TABLE_NAME = "fhvhv_tripdata"
    TAXI_TYPE_ID = 4

In [ ]:
class TaxiZonesSilverTable(SilverTable):

    TABLE_NAME = "nyc_taxi_zones"

In [ ]:
class OpenAQSilverTable(SilverTable, TableWithDate, ABC):

    DATE_COLUMN = "measurement_datetime"

In [ ]:
class OpenAQHourlyTable(OpenAQSilverTable):

    TABLE_NAME = "openaq_measurements_hours"

In [ ]:
class OpenAQDailyTable(OpenAQSilverTable):

    TABLE_NAME = "openaq_measurements_days"

In [ ]:
class FXSilverTable(SilverTable):

    TABLE_NAME = "usd_eur_fx_rates"

In [ ]:
class GDPSilverTable(SilverTable):

    TABLE_NAME = "usa_gdp_values"

In [ ]:
class GoldTable(Table, ABC):

    STORAGE_NAME = "Gold_warehouse"

    def read(self):
        return spark.read.synapsesql(f"{self.STORAGE_NAME}.dbo.{self.TABLE_NAME}")
    
    def append(self, new_rows):
        new_rows.write.mode("append").synapsesql(f"{self.STORAGE_NAME}.dbo.{self.TABLE_NAME}")


In [ ]:
class DimDate(GoldTable, TableWithPKS):

    TABLE_NAME = "DimDate"
    PKS = ["date"]

In [ ]:
class DimZone(GoldTable, TableWithPKS):

    TABLE_NAME = "DimZone"
    PKS = ["LocationID"]

In [ ]:
class DimTaxiType(GoldTable, TableWithPKS):

    TABLE_NAME = "DimTaxiType"
    PKS = ["taxi_type_id"]